# Final test: lighting breakdown on half B (real lighting, no GPU)

**CRISP-DM stage:** Evaluation

The locked Phase 7 notebook `07_lighting`, with **one change: it reads the half B photo scores** of the Final test instead of the half A ones. The method is the same, as the Lock requires.

**Two questions:**
1. **Within one lighting, can the model still rank defective photos above good ones?** Image AU-ROC using only the photos taken under that lighting.
2. **The factory question:** if the pass-or-fail line is set just above the highest good photo under `regular` light, and the light then changes, what share of good parts is thrown away? This line is a way of measuring, not a choice: it is worked out inside half B and is not the locked line.

**Limits, stated up front:** each lighting has only 10 half B photos, from 2 good parts and 8 defective parts. These numbers are shaky and only show direction. This is real lighting from the public test set, not the unseen lighting of the private `test_private_mixed` set. C6 at 512 is left out, as in Phase 7.

**Runs on:** the laptop.


In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Finds the project folder, whether the notebook runs from the project or from notebooks/
PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "src").exists():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
sys.path.insert(0, str(PROJECT_FOLDER))
from src import evaluate

# Final test: the half B files of C1 to C8. C6 at 512 is in its own folder, so it is left out
RESULT_FOLDERS = []
for config_id in ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"]:
    RESULT_FOLDERS.append(PROJECT_FOLDER / "outputs/results/final_test/half_b" / config_id)
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs/results/final_test/lighting"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
FIGURE_FOLDER = PROJECT_FOLDER / "outputs/figures"
PRODUCTS = ["sheet_metal", "vial"]
FIGURE_DPI = 150


def lighting_of(file_name):
    """Return the lighting part of a name like 007_shift_2.png, here shift_2."""
    stem = Path(file_name).stem
    first_underscore = stem.index("_")
    return stem[first_underscore + 1:]


def read_scores(score_file):
    """Return file names, labels (True for defective) and scores from one run's file."""
    names = []
    labels = []
    scores = []
    with open(score_file, encoding="utf-8") as file:
        for row in csv.DictReader(file):
            names.append(row["file_name"])
            labels.append(row["label"] == "bad")
            scores.append(float(row["score"]))
    return names, torch.tensor(labels), torch.tensor(scores)


# Every run's photo-score file, grouped by configuration and product
runs = {}
for folder in RESULT_FOLDERS:
    if not folder.exists():
        continue
    for score_file in sorted(folder.rglob("*_photo_scores.csv")):
        # A name like C5_sheet_metal_seed42_photo_scores.csv
        config_id = score_file.name.split("_")[0]
        product = None
        for product_name in PRODUCTS:
            if score_file.name.startswith(config_id + "_" + product_name + "_"):
                product = product_name
        key = (config_id, product)
        if key not in runs:
            runs[key] = []
        runs[key].append(score_file)
for key in sorted(runs):
    print(key, len(runs[key]), "run(s)")

## 1. Measure each lighting

For each run, and each lighting:
- **AU-ROC within the lighting**, from only that lighting's photos.
- **Good parts thrown away:** the line sits just above the highest-scoring good `regular` photo. The share of good photos under this lighting that score above it is the false-reject rate when the light changes. Under `regular` itself, it is 0 by construction.
- **Defects caught** at the same line, for balance.

PatchCore has three seeds, so each number is the average over its seeds.

In [ ]:
rows = []
for key in sorted(runs):
    config_id, product = key
    per_lighting = {}
    for score_file in runs[key]:
        names, labels, scores = read_scores(score_file)
        lightings = []
        for name in names:
            lightings.append(lighting_of(name))
        # The line: just above the highest good photo under regular light
        regular_good_scores = []
        for position in range(len(names)):
            if lightings[position] == "regular" and not labels[position]:
                regular_good_scores.append(float(scores[position]))
        line = max(regular_good_scores)
        for lighting in sorted(set(lightings)):
            chosen = []
            for position in range(len(names)):
                if lightings[position] == lighting:
                    chosen.append(position)
            chosen = torch.tensor(chosen)
            lighting_labels = labels[chosen]
            lighting_scores = scores[chosen]
            auroc = evaluate.image_auroc(lighting_scores, lighting_labels)
            good_rejected = float((lighting_scores[~lighting_labels] > line).float().mean())
            defects_caught = float((lighting_scores[lighting_labels] > line).float().mean())
            if lighting not in per_lighting:
                per_lighting[lighting] = {"auroc": [], "good_rejected": [], "defects_caught": [],
                                          "good": int((~lighting_labels).sum()), "defective": int(lighting_labels.sum())}
            per_lighting[lighting]["auroc"].append(auroc)
            per_lighting[lighting]["good_rejected"].append(good_rejected)
            per_lighting[lighting]["defects_caught"].append(defects_caught)
    for lighting in per_lighting:
        values = per_lighting[lighting]
        seeds = len(values["auroc"])
        rows.append({
            "config": config_id,
            "product": product,
            "lighting": lighting,
            "good_photos": values["good"],
            "defective_photos": values["defective"],
            "seeds": seeds,
            "auroc_within_lighting": round(sum(values["auroc"]) / seeds, 4),
            "good_rejected_at_regular_line": round(sum(values["good_rejected"]) / seeds, 4),
            "defects_caught_at_regular_line": round(sum(values["defects_caught"]) / seeds, 4),
        })

with open(OUTPUT_FOLDER / "lighting_by_config.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

for product in PRODUCTS:
    print("===", product)
    print("config  lighting       AU-ROC  good thrown away  defects caught")
    for row in rows:
        if row["product"] == product:
            print(row["config"].ljust(7), row["lighting"].ljust(14), str(row["auroc_within_lighting"]).ljust(7),
                  str(row["good_rejected_at_regular_line"]).ljust(17), row["defects_caught_at_regular_line"])

## 2. Slope chart: regular against changed lighting

Each line is one configuration. Its left end is AU-ROC under `regular` light, and its right end is the average AU-ROC over all changed lightings. A line going down means the method ranks worse when the light changes. The second panel does the same for good parts thrown away at the regular-light line.

In [ ]:
def summary_for(product, measure):
    """Return {config: (regular value, average over changed lightings)} for one measure."""
    summary = {}
    for row in rows:
        if row["product"] != product:
            continue
        if row["config"] not in summary:
            summary[row["config"]] = {"regular": None, "changed": []}
        if row["lighting"] == "regular":
            summary[row["config"]]["regular"] = row[measure]
        else:
            summary[row["config"]]["changed"].append(row[measure])
    pairs = {}
    for config_id in summary:
        changed = summary[config_id]["changed"]
        pairs[config_id] = (summary[config_id]["regular"], sum(changed) / len(changed))
    return pairs


measures = [("auroc_within_lighting", "Image AU-ROC within one lighting"),
            ("good_rejected_at_regular_line", "Good parts thrown away\n(line set under regular light)")]
# One fixed colour per configuration, the same in every panel
COLOURS = {"C1": "tab:blue", "C2": "tab:orange", "C3": "tab:cyan", "C4": "tab:green",
           "C5": "tab:red", "C6": "tab:olive", "C7": "tab:purple", "C8": "tab:brown"}
figure, axes = plt.subplots(len(measures), len(PRODUCTS), figsize=(10, 9), squeeze=False)
for column in range(len(PRODUCTS)):
    product = PRODUCTS[column]
    for row_number in range(len(measures)):
        measure, label = measures[row_number]
        axis = axes[row_number][column]
        pairs = summary_for(product, measure)
        # Labels sorted by height, and pushed apart when closer than 0.05, so they never overlap
        label_heights = {}
        last_height = None

        def changed_value_of(name):
            return pairs[name][1]

        for config_id in sorted(pairs, key=changed_value_of):
            height = pairs[config_id][1]
            if last_height is not None and height < last_height + 0.05:
                height = last_height + 0.05
            label_heights[config_id] = height
            last_height = height
        for config_id in sorted(pairs):
            regular_value, changed_value = pairs[config_id]
            axis.plot([0, 1], [regular_value, changed_value], marker="o", color=COLOURS[config_id])
            axis.annotate(config_id, (1.04, label_heights[config_id]), va="center", fontsize=9, color=COLOURS[config_id])
        axis.set_xticks([0, 1])
        axis.set_xticklabels(["regular", "changed (average)"])
        axis.set_xlim(-0.2, 1.3)
        axis.set_ylim(-0.05, 1.05)
        axis.set_ylabel(label)
        axis.set_title(product.replace("_", " ").title())
        axis.grid(axis="y", alpha=0.3)
figure.suptitle("Half B, real lighting from the public test set. 10 photos per lighting, so values are shaky", y=1.0)
figure.tight_layout()
figure.savefig(FIGURE_FOLDER / "final_lighting_slope.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()